In [1]:
 !pip install dataset

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 19.0 MB/s eta 0:00:00
  Attempting uninstall: sqlalchemy
    Found existing installation: SQLAlchemy 2.0.48
    Uninstalling SQLAlchemy-2.0.48:
      Successfully uninstalled SQLAlchemy-2.0.48
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.28.0 requires sqlalchemy<3.0.0,>=2.0, but you have sqlalchemy 1.4.54 which is incompatible.
ipython-sql 0.5.0 requires sqlalchemy>=2.0, but you have sqlalchemy 1.4.54 which is incompatible.


In [2]:
!pip install transformers

In [ ]:
from huggingface_hub import login

login("") 
#the key is in .env


In [ ]:
from datasets import load_dataset
dataset = load_dataset("imdb")

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
  return tokenizer(examples["text"], padding="max_length", truncation=True)  #the dataset has key called"text"

tokenized_dataset = dataset.map(tokenize_function, batched = True)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [6]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50000
    })
})

In [7]:
tokenized_dataset['train'][0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [8]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 45.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
!pip uninstall -y transformers peft accelerate trl flash-attn
!pip install transformers==4.41.2 datasets==2.19.1 accelerate==0.30.1 peft==0.11.1

In [9]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="results",
    evaluation_strategy="epoch",   # correct
    learning_rate=2e-5,
    per_device_train_batch_size=16,  #training on 16 examples at once
    per_device_eval_batch_size=16,  #how many examples the model only checks/predicts at once
    num_train_epochs=1,
    weight_decay=0.1
)

ImportError: cannot import name 'is_flash_attn_4_available' from 'transformers.utils' (/usr/local/lib/python3.12/dist-packages/transformers/utils/__init__.py)

In [ ]:
from transformers import AutoModelForSequenceClassification, Trainer

#loading pre trained model
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)  #since 2 labels positive and negative sentiment

#initialize the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset = tokenized_dataset['train'],
    eval_dataset = tokenized_dataset['test']
)


Training the model

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"  #setting this as it asks to create wandb account

In [ ]:
trainer.train()

evaluate the model

In [ ]:
results = trainer.evaluate()
print(results)